In [33]:
import os

# Adjust the path to match your Box sync folder location
box_path = os.path.expanduser(r"C:\Users\HIALAB\Box\Human_AGV_project\ISU_Modeling\Code")

In [35]:
# Print the current working directory
print(os.getcwd())

C:\Users\HIALAB\Box\Human_AGV_project\ISU_Modeling\Code\human-agv-interaction


In [37]:
# base libraries
import numpy as np
import pandas as pd
# regular expression module in python to find all sequences of digits in a given string
import re

import os
from datetime import datetime, timedelta
import math
from scipy import stats

from frechetdist import frdist
import sys
from concurrent.futures import ThreadPoolExecutor
from typing import List, Tuple

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

In [39]:
import import_ipynb
from derived_classes import PerInteractionDataProcessor

In [41]:
per_interaction_data = pd.read_csv(os.path.join(box_path, "per_interaction_data.csv"))
pre_survey = pd.read_csv(os.path.join(box_path, "pre_survey.csv"))
post_survey = pd.read_csv(os.path.join(box_path, "post_survey.csv"))
cross_first = pd.read_csv(os.path.join(box_path, 'csvfiles', 'cross_first.txt'))
study_data_processed = pd.read_csv(os.path.join(box_path, 'study_data_processed.csv'))
# If you decide to do the data processing in multiple steps, save the data, and read it each time
# per_interaction_data_merged = pd.read_csv(os.path.join(box_path, 'per_interaction_data_merged.csv'))

In [ ]:
# Making the format of PIDs as 001, 002, 003, ...
pre_survey['PID'] = pre_survey['PID'].astype(str).apply(lambda x: x.zfill(3))
post_survey['PID'] = post_survey['PID'].astype(str).apply(lambda x: x.zfill(3))
per_interaction_data['PID'] = per_interaction_data['PID'].apply(lambda x: str(x).zfill(3))
study_data_processed['PID'] = study_data_processed['PID'].apply(lambda x: str(x).zfill(3))

In [ ]:
# Check the data types of 'DRate' and 'PID' columns in both DataFrames
print("pre_survey data types:")
print(pre_survey[['PID']].dtypes)

print("\nper_interaction_data data types:")
print(per_interaction_data[['DRate', 'PID', 'AGVname']].dtypes)

print("\npost_survey data types:")
print(post_survey[['DRate', 'PID']].dtypes)

# If needed, convert the data types to ensure consistency
per_interaction_data['DRate'] = per_interaction_data['DRate'].astype(str)
per_interaction_data['PID'] = per_interaction_data['PID'].astype(str)

post_survey['DRate'] = post_survey['DRate'].astype(str)
post_survey['PID'] = post_survey['PID'].astype(str)

## Merge Pre, Post, and Study DataFrames 

In [ ]:
# Add new columns with empty strings
new_columns = {
    'Age': "", 'Gender': "", 'Ethnicity': "", 'GamingFrequency': "", 'VRExperience': "", 
    'VRHeadsetExperience': "", 'AGVInteraction': "", 'PerfectAutomation': "", 
    'TrustPropensity': "", 'AutomationExperience': "", 'Trust1': "", 'Trust2': "", 
    'MWL': "", 'Assessment': "",
}

# Add the new columns to the DataFrame
per_interaction_data = per_interaction_data.assign(**new_columns)

In [ ]:
per_interaction_data.head()

In [ ]:
per_interaction_data.columns

### 'Age', 'Gender', 'Ethnicity', 'GamingFrequency', 'VRExperience', 'VRHeadsetExperience', 'AGVInteraction', 'PerfectAutomation', 'TrustPropensity', and 'AutomationExperience' 

In [ ]:
# Create the dictionary mapping only the desired columns
pre_dict = pre_survey.set_index('PID')[['Age', 'Gender', 'Ethnicity', 'GamingFrequency', 'VRExperience',
                                         'VRHeadsetExperience', 'AGVInteraction', 'PerfectAutomation',
                                         'TrustPropensity', 'AutomationExperience']].to_dict()

# Map the values from pre_survey to per_interaction_data based on the unique 'PID'
for col in pre_dict.keys():
    per_interaction_data[col] = [pre_dict[col].get(pid) for pid in per_interaction_data['PID']]

### 'Trust1', 'Trust2', 'MWL', 'Assessment' 

In [ ]:
# Create the dictionary mapping only the desired columns
post_dict = post_survey.set_index(['DRate', 'PID'])[['Trust1', 'Trust2', 'MWL', 'Assessment']].to_dict()

# Map the values from post_survey to per_interaction_data based on the unique 'PID' and 'DRate'
for col in post_dict.keys():
    per_interaction_data[col] = [post_dict[col].get((drate, pid)) for drate, pid in per_interaction_data[['DRate', 'PID']].values]

In [ ]:
per_interaction_data.head()

In [ ]:
per_interaction_data['StartTime'] = pd.to_datetime(per_interaction_data['StartTime'], errors='coerce', format='%H:%M:%S')
per_interaction_data['EndTime'] = pd.to_datetime(per_interaction_data['EndTime'], errors='coerce', format='%H:%M:%S')

In [ ]:
per_interaction_data = per_interaction_data.drop(per_interaction_data[per_interaction_data['PID'].isin(['025'])].index)

In [ ]:
processor_for_per_interaction_data = PerInteractionDataProcessor(per_interaction_data, study_data_processed)
high_first_low_last = ['002', '004', '006', '008', '010', '012', '014', '016', '020', 
                       '022', '027', '029', '031', '033', '035', '037', '039', '041', 
                       '043', '045']

low_first_high_last = ['001', '003', '007', '009', '011', '013', '015', '017', '019', 
                       '021', '024', '026', '028', '030', '032', '034', '036', '038', 
                       '040', '042', '044', '046']
for element in low_first_high_last:
    print(type(element))
    
per_interaction_data_merged = processor_for_per_interaction_data.process(high_first_low_last, low_first_high_last, cross_first)

In [ ]:
per_interaction_data_merged.iloc[0:30]

## Mismatch Checking 
### Checking for mismatches between unique values of columns in two datasets

In [ ]:
# Function to find mismatches between unique values of columns in two datasets
def check_column_mismatches(study_data_processed, per_interaction_data, columns):
    for col in columns:
        study_unique = study_data_processed[col].unique()
        interaction_unique = per_interaction_data[col].unique()

        # Find mismatches in both directions
        study_not_in_interaction = set(study_unique) - set(interaction_unique)
        interaction_not_in_study = set(interaction_unique) - set(study_unique)

        # Print the results for the current column
        print(f"\nColumn: {col}")
        if study_not_in_interaction:
            print(f"Values in 'study_data_processed' but not in 'per_interaction_data': {study_not_in_interaction}")
        else:
            print("No mismatches found from 'study_data_processed' to 'per_interaction_data'.")

        if interaction_not_in_study:
            print(f"Values in 'per_interaction_data' but not in 'study_data_processed': {interaction_not_in_study}")
        else:
            print("No mismatches found from 'per_interaction_data' to 'study_data_processed'.")

# List of columns to compare
columns_to_check = ['DRate', 'PID', 'AGVname']

# Check for mismatches between the two datasets
check_column_mismatches(study_data_processed, per_interaction_data_merged, columns_to_check)

In [ ]:
processor_for_per_interaction_data_merged = PerInteractionDataProcessor(per_interaction_data, study_data)

per_interaction_data_merged = processor_for_per_interaction_data_merged.process(high_first_low_last, low_first_high_last, cross_first)

In [ ]:
per_interaction_data_merged.iloc[20:40]

In [ ]:
per_interaction_data_merged.columns

In [ ]:
null_counts = per_interaction_data_merged.isnull().sum()
print(null_counts)

In [28]:
file_name = "per_interaction_data_merged.csv"  # Specify your desired file name
full_path = os.path.join(box_path, file_name)

# Save the DataFrame as a CSV file
per_interaction_data_merged.to_csv(full_path, index=False)

### Cross First

In [14]:
per_interaction_data_merged = pd.read_csv(os.path.join(box_path, "per_interaction_data_merged.csv"))

In [20]:
# Ensure PID is of the correct type
cross_first['PID'] = cross_first['PID'].astype(int).astype(str).str.zfill(3)
per_interaction_data_merged['PID']=per_interaction_data_merged['PID'].astype(int).astype(str).str.zfill(3)

# Merge the two DataFrames on the common columns
merged_data = per_interaction_data_merged.merge(
    cross_first[['PID', 'DRate', 'AGVname', 'cross_first']],
    on=['PID', 'DRate', 'AGVname'],
    how='left'
)

# Update the Cross_First column in per_interaction_data
per_interaction_data_merged['Cross_First'] = merged_data['cross_first']

In [26]:
per_interaction_data_merged.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,AGV_Path_Complexity,Trust_before,Cross_First_Flag,User_Relative_Speed,AGV_Relative_Speed,Frechet_Distance_3,Frechet_Distance_5,Frechet_Distance_7,Frechet_Distance_10,Cross_First
0,001,High,2024-5-3,14,28,53,1,1900-01-01 15:02:48,1900-01-01 15:03:38,219,...,Straight,103.457506,True,74.580213,100.420508,730.13,1188.92,1532.83,1771.61,N\A
1,001,High,2024-5-3,14,28,53,2,1900-01-01 15:03:47,1900-01-01 15:04:29,0,...,Complex,211.137767,True,47.888087,232.345812,14.56,97.05,108.21,247.34,N\A
2,001,High,2024-5-3,14,28,53,3,1900-01-01 15:04:37,1900-01-01 15:05:34,268,...,Complex,211.137767,True,63.130599,120.176130,282.09,482.54,821.32,1067.08,User
3,001,High,2024-5-3,14,28,53,4,1900-01-01 15:05:46,1900-01-01 15:06:37,779,...,Complex,211.137767,True,43.571998,263.284416,421.34,544.15,577.55,610.84,User
4,001,High,2024-5-3,14,28,53,5,1900-01-01 15:06:41,1900-01-01 15:07:44,868,...,Complex,147.796437,True,93.720008,91.614207,486.65,852.85,1225.51,1722.14,N\A


### Trust Before

In [15]:
per_interaction_data_merged = pd.read_csv(os.path.join(box_path, "per_interaction_data_merged.csv"))
model_data_for_transition_matrix = pd.read_csv(os.path.join(box_path, "model_data_for_transition_matrix.csv"))

In [17]:
model_data_for_transition_matrix.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,Assessment,Frechet_Distance,Gaze_on_AGV,User_Relative_Speed,User_Trajectory,AGV_Approaching,AGV_User_Combination,AGV_Path_Complexity,Interaction No,Trust_before
0,1,High,2024-5-3,14,28,53,1,15:2:48,15:3:38,219,...,1.812,NaN,0.063,137.683,Diagonal,North,North - Diagonal,Complex,1,4.9
1,1,High,2024-5-3,14,28,53,2,15:3:47,15:4:29,0,...,1.812,NaN,0.038,109.577,Straight,South,South - Straight,Straight,2,10.0
2,1,High,2024-5-3,14,28,53,3,15:4:37,15:5:34,268,...,1.812,NaN,0.069,131.592,Diagonal,South,South - Diagonal,Complex,3,10.0
3,1,High,2024-5-3,14,28,53,4,15:5:46,15:6:37,779,...,1.812,NaN,0.231,95.205,Straight,Northeast,Northeast - Straight,Complex,4,10.0
4,1,High,2024-5-3,14,28,53,5,15:6:41,15:7:44,868,...,1.812,NaN,0.234,185.999,Straight,Northwest,Northwest - Straight,Complex,5,7.0


In [21]:
per_interaction_data_merged.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,AGV_Path_Complexity,Trust_before,Cross_First_Flag,User_Relative_Speed,AGV_Relative_Speed,Frechet_Distance_3,Frechet_Distance_5,Frechet_Distance_7,Frechet_Distance_10,Cross_First
0,1,High,2024-5-3,14,28,53,1,1900-01-01 15:02:48,1900-01-01 15:03:38,219,...,Straight,103.457506,True,74.580213,100.420508,730.13,1188.92,1532.83,1771.61,N\A
1,1,High,2024-5-3,14,28,53,2,1900-01-01 15:03:47,1900-01-01 15:04:29,0,...,Complex,211.137767,True,47.888087,232.345812,14.56,97.05,108.21,247.34,N\A
2,1,High,2024-5-3,14,28,53,3,1900-01-01 15:04:37,1900-01-01 15:05:34,268,...,Complex,211.137767,True,63.130599,120.176130,282.09,482.54,821.32,1067.08,User
3,1,High,2024-5-3,14,28,53,4,1900-01-01 15:05:46,1900-01-01 15:06:37,779,...,Complex,211.137767,True,43.571998,263.284416,421.34,544.15,577.55,610.84,User
4,1,High,2024-5-3,14,28,53,5,1900-01-01 15:06:41,1900-01-01 15:07:44,868,...,Complex,147.796437,True,93.720008,91.614207,486.65,852.85,1225.51,1722.14,N\A


In [19]:
per_interaction_data_merged['PID'] = per_interaction_data_merged['PID'].astype(int)
per_interaction_data_merged['DRate'] = per_interaction_data_merged['DRate'].astype(str)
per_interaction_data_merged['AGVname'] = per_interaction_data_merged['AGVname'].astype(int)

model_data_for_transition_matrix['PID'] = model_data_for_transition_matrix['PID'].astype(int)
model_data_for_transition_matrix['DRate'] = model_data_for_transition_matrix['DRate'].astype(str)
model_data_for_transition_matrix['AGVname'] = model_data_for_transition_matrix['AGVname'].astype(int)

In [25]:
# Merge to bring Trust_Before values from model_data_for_transition_matrix
per_interaction_data_merged = per_interaction_data_merged.merge(
    model_data_for_transition_matrix[['PID', 'DRate', 'AGVname', 'Trust_before']],
    on=['PID', 'DRate', 'AGVname'],
    how='left',  # Keeps all rows from per_interaction_data_merged
    suffixes=('', '_new')
)

# Replace the Trust_Before column in per_interaction_data_merged with the new values
per_interaction_data_merged['Trust_before'] = per_interaction_data_merged['Trust_before_new']

# Drop the temporary column
per_interaction_data_merged.drop(columns=['Trust_before_new'], inplace=True)

In [27]:
per_interaction_data_merged.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,AGV_Path_Complexity,Trust_before,Cross_First_Flag,User_Relative_Speed,AGV_Relative_Speed,Frechet_Distance_3,Frechet_Distance_5,Frechet_Distance_7,Frechet_Distance_10,Cross_First
0,1,High,2024-5-3,14,28,53,1,1900-01-01 15:02:48,1900-01-01 15:03:38,219,...,Straight,4.9,True,74.580213,100.420508,730.13,1188.92,1532.83,1771.61,N\A
1,1,High,2024-5-3,14,28,53,2,1900-01-01 15:03:47,1900-01-01 15:04:29,0,...,Complex,10.0,True,47.888087,232.345812,14.56,97.05,108.21,247.34,N\A
2,1,High,2024-5-3,14,28,53,3,1900-01-01 15:04:37,1900-01-01 15:05:34,268,...,Complex,10.0,True,63.130599,120.176130,282.09,482.54,821.32,1067.08,User
3,1,High,2024-5-3,14,28,53,4,1900-01-01 15:05:46,1900-01-01 15:06:37,779,...,Complex,10.0,True,43.571998,263.284416,421.34,544.15,577.55,610.84,User
4,1,High,2024-5-3,14,28,53,5,1900-01-01 15:06:41,1900-01-01 15:07:44,868,...,Complex,7.0,True,93.720008,91.614207,486.65,852.85,1225.51,1722.14,N\A


In [29]:
file_name = "per_interaction_data_merged.csv"  
full_path = os.path.join(box_path, file_name)

per_interaction_data_merged.to_csv(full_path, index=False)